In [ ]:
# --- Install all required packages for the project ---
%pip install pandas geopandas rasterio shapely fiona pyproj rapidfuzz xarray netCDF4 dask matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 72.1 MB/s eta 0:00:00


In [ ]:
# In Notebook 00, REPLACE the first cell with this corrected version.

# --- CELL 1: Imports and Main Configuration ---

# --- CORE IMPORTS ---
import os
from google.colab import drive
import re, unicodedata, math, numpy as np, pandas as pd, geopandas as gpd, xarray as xr
import matplotlib.pyplot as plt, seaborn as sns
from rapidfuzz import process, fuzz
from rasterio import features
from rasterio.transform import from_origin
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut

# --- UNIVERSAL FILE PATHS AND SETTINGS ---
PROJECT_DIR = "/content/drive/MyDrive/IndiaProject_Colab"

# Raw Data Files
RAW_DIR = os.path.join(PROJECT_DIR, "data", "raw")
CSV_CROP = os.path.join(RAW_DIR, "ICRISAT-District Level Data_Apportioned.csv")
# <<< NEW: Add the Irrigation Data CSV path >>>
CSV_IRRIG = os.path.join(RAW_DIR, "ICRISAT-District Level Irrigation Data.csv")

SHP_DIST = os.path.join(RAW_DIR, "icrisat_boundary_match.shp")
# ... (Climate files remain the same) ...
SM_FILE = os.path.join(RAW_DIR, "sm_data_00_23.nc")
TX_FILE = os.path.join(RAW_DIR, "tmax_data_00_23.nc")
PR_FILE = os.path.join(RAW_DIR, "precip_data.nc")
SIF_FILE = os.path.join(RAW_DIR, "csif_data_updated.nc")

# <<< --- THE FIX IS HERE --- >>>
# CROPGRIDS file paths are now pointing directly to the 'raw' folder.
CROP_GRID_FILES = {
    "maize": os.path.join(RAW_DIR, "CROPGRIDSv1.08_maize.nc"),
    "rice": os.path.join(RAW_DIR, "CROPGRIDSv1.08_rice.nc"),
    "sorghum_kharif": os.path.join(RAW_DIR, "CROPGRIDSv1.08_sorghum.nc"),
    "sorghum_rabi": os.path.join(RAW_DIR, "CROPGRIDSv1.08_sorghum.nc"),
}
# --- END OF FIX ---

SM_VAR, TX_VAR, PR_VAR = "sm", "tmax", "precip"

# Processed Data Files
PROC_DIR = os.path.join(PROJECT_DIR, "data", "processed")
DIST_GPKG = os.path.join(PROC_DIR, "districts_joined.gpkg")

# <<< CHANGED: Update the Processed Filename to reflect new content >>>
YLD_FILE = os.path.join(PROC_DIR, "icrisat_yield_area_irrig_4crops_long_2000_2017.csv")

# Results Directory
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")

# --- Other Settings ---
PR_IS_RATE_MM_PER_DAY = True
ALL_TOUCHED = True
FEATURES = ["sm_season_mean", "pr_season_total", "tmax_season_mean", "tmax_season_max"]
YCOL = "yield_kg_per_ha"
YEAR_MIN, YEAR_MAX = 2000, 2017
MIN_YEARS_FOR_MODEL = 15

CROP_CALENDARS = {
    "maize": {"planting_start": 3, "planting_end": 7, "growing_start":  8, "growing_end":  8, "harvest_start": 9, "harvest_end": 12},
    "rice": {"planting_start": 5, "planting_end": 7, "growing_start":  8, "growing_end":  8, "harvest_start": 9, "harvest_end": 12},
    "sorghum_kharif": {"planting_start": 6, "planting_end": 8, "growing_start":  9, "growing_end":  10, "harvest_start": 11, "harvest_end": 12},
    "sorghum_rabi": {"planting_start": 10, "planting_end": 11, "growing_start":  12, "growing_end":  1, "harvest_start": 2, "harvest_end": 4}
}

In [ ]:
# --- CELL 2: Helper Functions (Geospatial, Text, Date) ---

def _norm(s: str) -> str:
    if pd.isna(s): return ""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii","ignore").decode("ascii")
    s = s.lower().replace("&","and").replace(".","")
    s = re.sub(r"\bdistrict\b","", s)
    s = re.sub(r"\s+"," ", s).strip()
    return s

STATE_ALIAS = { "nct of delhi": "delhi", "orissa": "odisha", "uttaranchal": "uttarakhand", "pondicherry": "puducherry", "jammu and kashmir": "jammu kashmir" }
def canon_state(s: str) -> str:
    s = _norm(s)
    return STATE_ALIAS.get(s, s)

def find_first(cols, candidates):
    for c in candidates:
        if c in cols: return c
    return None

def _split_variants(val):
    if pd.isna(val): return []
    parts = re.split(r"[,;/()|]+", str(val))
    return sorted(list(set([_norm(p) for p in parts if _norm(p)])))

def get_month_range(start, end):
    if start <= end: return list(range(start, end + 1))
    else: return list(range(start, 13)) + list(range(1, end + 1))

def get_middle_month(start, end):
    if start <= end: return int((start + end) / 2 + 0.5)
    else:
        months = get_month_range(start, end)
        return months[len(months) // 2]

In [ ]:
# --- CELL 3: Updated Climate and SIF Aggregation (Handles Year-Crossing Seasons) ---

def _to_monthly_time(da: xr.DataArray, ds: xr.Dataset) -> xr.DataArray:
    """Standardizes time dimensions to datetime objects."""
    dims = [d.lower() for d in da.dims]
    if "time" in dims:
        t = pd.to_datetime((da["time"] if "time" in da.coords else ds["time"]).values)
        t = pd.to_datetime([f"{d.year:04d}-{d.month:02d}-01" for d in t])
        return da.assign_coords(time=("time", t)).transpose("time", ...)

    # Handle cases where time is split into Year/Month integers (common in some SIF datasets)
    y = next((d for d in da.dims if d.lower()=="year"), None)
    m = next((d for d in da.dims if d.lower()=="month"), None)
    years  = np.array((da[y] if y in da.coords else ds[y]).values, int)
    months = np.arange(1, da.sizes[m]+1, dtype=int)
    time = np.array([np.datetime64(f"{int(Y):04d}-{int(M):02d}-01") for Y in years for M in months])
    with pd.option_context("future.no_silent_downcasting", True):
        return da.transpose(y, m, ...).stack(time=(y, m)).assign_coords(time=("time", time)).transpose("time", ...)

def _load_districts(gpkg_path=DIST_GPKG):
    gdf = gpd.read_file(gpkg_path).dropna(subset=["Dist Code"])
    gdf = gdf.to_crs(4326).dissolve(by="Dist Code", as_index=False, aggfunc="first")
    gdf["Dist Code"] = gdf["Dist Code"].astype(int)
    return gdf

# --- 1. CLIMATE AGGREGATION (With "Smart Season" Logic) ---
def get_climate_panel_for_months_NO_WEIGHTING(months: list[int], year_min=YEAR_MIN, year_max=YEAR_MAX):
    print(f"--- Running Climate Aggregation (Smart Season: {months}) ---")

    # DETECT YEAR CROSSING: If the month list wraps (e.g., [10, 11, 12, 1, 2])
    # The 'start' of the season is the first month in the list.
    season_start_month = months[0]
    is_year_crossing = any(m < season_start_month for m in months)

    if is_year_crossing:
        print(f"    [!] Detected Year-Crossing Season (Start: {season_start_month}). adjusting logic...")

    def process_climate_nc(kind: str):
        # Load Data
        if kind == "sm":
            ds = xr.open_dataset(SM_FILE); da = ds[SM_VAR]
        elif kind == "tmax":
            ds = xr.open_dataset(TX_FILE); da = ds[TX_VAR]

        # Standardize Time
        da_m = _to_monthly_time(da, ds).sel(time=slice(f"{year_min-1}-01-01", f"{year_max+1}-12-31"))

        # Unit Conversion for Temp
        if kind == "tmax" and (da_m.attrs.get("units") or "").lower() in {"k", "kelvin"}:
            da_m -= 273.15

        # --- SMART SEASON GROUPING ---
        # 1. Filter to keep only relevant months
        da_subset = da_m.where(da_m['time.month'].isin(months), drop=True)

        # 2. Assign "Season Year"
        # If the season crosses the year (e.g. Rabi), any month < start_month belongs to the PREVIOUS crop year.
        # Ex: Season Start=10. Jan(1) 2013 belongs to Season 2012.
        # If no crossing (Rice), season_year == calendar_year.

        if is_year_crossing:
            # Shift Jan-May months BACK one year
            # "time.month < season_start_month" captures the tail end of the season
            season_years = da_subset['time.year'].copy()
            # Logic: If month is small (Jan/Feb) and season starts late (Oct), shift year -1
            da_subset = da_subset.assign_coords(
                season_year=xr.where(da_subset['time.month'] < season_start_month,
                                     da_subset['time.year'] - 1,
                                     da_subset['time.year'])
            )
            # Group by our new custom coordinate
            seasonal = da_subset.groupby('season_year').mean('time')
            return seasonal.rename({'season_year': 'year_dim'})
        else:
            # Standard grouping
            seasonal = da_subset.groupby('time.year').mean('time')
            return seasonal.rename({'year': 'year_dim'})

    def district_panel_simple(seasonal_grid, gdf):
        if seasonal_grid is None: return pd.DataFrame(columns=['Dist Code', 'year'])
        # (Standard rasterize logic...)
        lats, lons = seasonal_grid.lat.values, seasonal_grid.lon.values
        dx, dy = float(np.abs(np.diff(lons)).mean()), float(np.abs(np.diff(lats)).mean())
        transform = from_origin(lons.min() - dx/2, lats.max() + dy/2, dx, dy)
        shapes = [(geom, int(code)) for geom, code in zip(gdf.geometry, gdf["Dist Code"])]
        mask = features.rasterize(shapes=shapes, out_shape=(len(lats), len(lons)), transform=transform, fill=0, dtype="int32", all_touched=True)
        if lats[0] < lats[-1]: mask = np.flipud(mask)
        results = []
        # Filter to requested range
        valid_years = [y for y in seasonal_grid.year_dim.values if year_min <= y <= year_max]

        for year_val in valid_years:
            da_slice = seasonal_grid.sel(year_dim=year_val)
            means = [da_slice.where(mask == code).mean().item(0) if da_slice.where(mask == code).notnull().any() else np.nan for code in gdf['Dist Code']]
            results.append(pd.DataFrame({'Dist Code': gdf['Dist Code'], 'year': year_val, 'value': means}))
        return pd.concat(results, ignore_index=True) if results else pd.DataFrame(columns=['Dist Code', 'year', 'value'])

    gdf = _load_districts()
    sm_panel = district_panel_simple(process_climate_nc('sm'), gdf).rename(columns={'value': 'sm_season_mean'})
    tmax_panel = district_panel_simple(process_climate_nc('tmax'), gdf).rename(columns={'value': 'tmax_season_mean'})
    return pd.merge(sm_panel, tmax_panel, on=['Dist Code', 'year'], how='outer')

# --- 2. SIF AGGREGATION (With "Smart Season" Logic) ---
def get_sif_panel_for_months(crop_name: str, months: list[int], year_min=YEAR_MIN, year_max=YEAR_MAX):
    print(f"--- Generating CROP-WEIGHTED SIF panel (Smart Season) for '{crop_name}' ---")
    try:
        # Load SIF (Point Data)
        sif_ds = xr.open_dataset(SIF_FILE, mask_and_scale=True)
        crop_grid_path = CROP_GRID_FILES[crop_name]
        crop_ds = xr.open_dataset(crop_grid_path)
    except Exception as e:
        print(f"ERROR loading files: {e}")
        return pd.DataFrame(columns=["Dist Code", "year", "csif_season_mean"])

    gdf_districts = _load_districts()

    # Detect Year Crossing
    season_start_month = months[0]
    is_year_crossing = any(m < season_start_month for m in months)

    all_years_results = []

    # We loop through "Season Years"
    for target_year in range(year_min, year_max + 1):

        # Construct the "Season Slice" manually
        # If Normal: 2012 [6,7,8] -> Get 2012 Data for months 6,7,8
        # If Crossing: 2012 [10,11,1,2] -> Get 2012 (10,11) AND 2013 (1,2)

        if not is_year_crossing:
            # Simple Case
            subset = sif_ds['csif'].sel(year=target_year)
            sif_seasonal = subset.where(subset['month'].isin(months)).mean("month")
        else:
            # Crossing Case (The Fix)
            # Part 1: Tail of current year (e.g., Oct-Dec 2012)
            months_part1 = [m for m in months if m >= season_start_month]
            subset1 = sif_ds['csif'].sel(year=target_year)
            data1 = subset1.where(subset1['month'].isin(months_part1))

            # Part 2: Head of next year (e.g., Jan-Mar 2013)
            months_part2 = [m for m in months if m < season_start_month]
            # Check if next year exists in data
            if (target_year + 1) in sif_ds.year:
                subset2 = sif_ds['csif'].sel(year=target_year + 1)
                data2 = subset2.where(subset2['month'].isin(months_part2))
                # Combine (Concat along time/month logic or simply average the values if we want seasonal mean)
                # Since SIF is point data here (lon, lat, month), we can concat and mean.
                combined = xr.concat([data1, data2], dim='month')
                sif_seasonal = combined.mean('month')
            else:
                # If we run off the edge of the dataset, skip
                continue

        # (Standard Spatial Weighting Logic...)
        sif_values_computed = sif_seasonal.compute().values
        df_points = pd.DataFrame({'lon': sif_ds['lon'].values, 'lat': sif_ds['lat'].values, 'sif_seasonal_mean': sif_values_computed})
        df_points['sif_seasonal_mean'] = df_points['sif_seasonal_mean'].replace(-9999, np.nan)
        df_points.dropna(subset=['sif_seasonal_mean'], inplace=True)
        if df_points.empty: continue

        lon_da = xr.DataArray(df_points['lon'], dims="location")
        lat_da = xr.DataArray(df_points['lat'], dims="location")
        weights = crop_ds['harvarea'].sel(lon=lon_da, lat=lat_da, method='nearest')
        df_points['crop_weight'] = weights.values
        df_points['crop_weight'] = df_points['crop_weight'].fillna(0).clip(lower=0)

        gdf_points = gpd.GeoDataFrame(df_points, geometry=gpd.points_from_xy(df_points.lon, df_points.lat), crs=gdf_districts.crs)
        joined_gdf = gpd.sjoin(gdf_points, gdf_districts, how="inner", predicate="within")
        if joined_gdf.empty: continue

        def weighted_avg(group):
            sif, weights = group['sif_seasonal_mean'], group['crop_weight']
            if weights.sum() == 0: return np.nan
            return (sif * weights).sum() / weights.sum()

        district_means = joined_gdf.groupby('Dist Code').apply(weighted_avg).reset_index(name='csif_season_mean')
        district_means['year'] = int(target_year)
        all_years_results.append(district_means)

    if not all_years_results: return pd.DataFrame(columns=["Dist Code", "year", "csif_season_mean"])
    final_panel = pd.concat(all_years_results, ignore_index=True)
    return final_panel

In [ ]:
# --- CELL REPLACEMENT: fit_district_model (Quadratic + Interaction) ---

def fit_district_model(g, features, ycol=YCOL):
    """
    Fits a 4-stage hierarchical model with QUADRATIC AND INTERACTION Climate terms.
    Hierarchy: Trend -> Climate (Lin + Quad + Inter) -> Irrigation -> SIF

    Includes:
    - SM, TMax
    - SM^2, TMax^2 (Centered)
    - SM * TMax (Centered Interaction)
    """
    # --- 1. Data Cleaning ---
    g = g.loc[(g["year"] >= YEAR_MIN) & (g["year"] <= YEAR_MAX)].copy()
    g = g[g[ycol] > 1.0]

    # Feature Groups
    feats_clim_linear = ["sm_season_mean", "tmax_season_mean"]
    feats_irrig = ["irrigated_pct"]
    feats_sif   = ["csif_season_mean"]

    # Check existence
    all_req_feats = feats_clim_linear + feats_irrig + feats_sif
    g = g.dropna(subset=[ycol] + all_req_feats)

    n = len(g)
    # Maintain strict threshold for complexity
    if n < MIN_YEARS_FOR_MODEL: return None, None

    # --- 2. Prepare Matrices ---
    y = g[ycol].to_numpy(float)

    # Trend
    years = g["year"].to_numpy(int)
    t = (years - years.mean())[:, None]

    # --- FEATURE ENGINEERING (Centered) ---
    # A. Linear Climate
    X_clim_lin = g[feats_clim_linear].to_numpy(float)

    # B. Centering (Crucial for Interactions)
    # We use the mean of the whole series for the main fit
    X_clim_centered = X_clim_lin - X_clim_lin.mean(axis=0)

    # C. Quadratic Terms (SM^2, TMax^2)
    X_clim_sq = X_clim_centered ** 2

    # D. Interaction Term (SM * TMax)
    # Element-wise multiplication of centered columns
    # Result is a single column shape (n,) -> reshape to (n,1)
    X_clim_inter = (X_clim_centered[:, 0] * X_clim_centered[:, 1])[:, None]

    # E. Combine Climate: [Linear (2), Quad (2), Inter (1)]
    X_clim_combined = np.hstack([X_clim_lin, X_clim_sq, X_clim_inter])

    # F. Other features
    X_irrig = g[feats_irrig].to_numpy(float)
    X_sif   = g[feats_sif].to_numpy(float)

    # --- Standardization ---
    scaler_clim = StandardScaler().fit(X_clim_combined)
    X_clim_sc = scaler_clim.transform(X_clim_combined)

    scaler_irrig = StandardScaler().fit(X_irrig)
    X_irrig_sc = scaler_irrig.transform(X_irrig)

    scaler_sif = StandardScaler().fit(X_sif)
    X_sif_sc = scaler_sif.transform(X_sif)

    # --- 3. The Hierarchy ---

    # Model 1: Trend
    mdl1 = LinearRegression().fit(t, y)
    r2_1 = mdl1.score(t, y)

    # Model 2: Trend + Climate (Linear + Quad + Inter)
    X2 = np.hstack([t, X_clim_sc])
    mdl2 = LinearRegression().fit(X2, y)
    r2_2 = mdl2.score(X2, y)

    # Model 3: Trend + Climate + Irrigation
    X3 = np.hstack([t, X_clim_sc, X_irrig_sc])
    mdl3 = LinearRegression().fit(X3, y)
    r2_3 = mdl3.score(X3, y)

    # Model 4: Trend + Climate + Irrigation + SIF
    X4 = np.hstack([t, X_clim_sc, X_irrig_sc, X_sif_sc])
    mdl4 = LinearRegression().fit(X4, y)
    r2_4 = mdl4.score(X4, y)

    # --- 4. Cross-Validation (Strict) ---
    logo = LeaveOneGroupOut()
    y_hat_cv = np.full(n, np.nan)

    # Raw linear features for CV loop
    X_raw_linear = np.hstack([X_clim_lin, X_irrig, X_sif])

    for train_idx, test_idx in logo.split(X_raw_linear, y, groups=years):
        # 1. Split Linear Data
        X_tr_lin = X_raw_linear[train_idx]
        X_te_lin = X_raw_linear[test_idx]

        # 2. Re-create Derived Terms using TRAIN Mean
        clim_tr = X_tr_lin[:, :2]
        clim_te = X_te_lin[:, :2]

        clim_mean_tr = clim_tr.mean(axis=0)

        # Center
        clim_tr_cen = clim_tr - clim_mean_tr
        clim_te_cen = clim_te - clim_mean_tr

        # Squares
        clim_tr_sq = clim_tr_cen ** 2
        clim_te_sq = clim_te_cen ** 2

        # Interaction
        inter_tr = (clim_tr_cen[:, 0] * clim_tr_cen[:, 1])[:, None]
        inter_te = (clim_te_cen[:, 0] * clim_te_cen[:, 1])[:, None]

        # 3. Stack features: [Lin(2), Quad(2), Inter(1), Irrig(1), SIF(1)]
        X_tr_final = np.hstack([clim_tr, clim_tr_sq, inter_tr, X_tr_lin[:, 2:]])
        X_te_final = np.hstack([clim_te, clim_te_sq, inter_te, X_te_lin[:, 2:]])

        # 4. Standardize (Fit on Train)
        scaler_cv = StandardScaler().fit(X_tr_final)
        X_tr_sc = scaler_cv.transform(X_tr_final)
        X_te_sc = scaler_cv.transform(X_te_final)

        # 5. Fit
        mdl_cv = LinearRegression().fit(np.hstack([t[train_idx], X_tr_sc]), y[train_idx])
        y_hat_cv[test_idx] = mdl_cv.predict(np.hstack([t[test_idx], X_te_sc]))

    mask = np.isfinite(y_hat_cv)
    r2_cv = np.nan
    if mask.sum() >= 2 and np.std(y[mask]) > 1e-9:
        sst_cv = np.sum((y[mask] - y[mask].mean())**2)
        sse_cv = np.sum((y[mask] - y_hat_cv[mask])**2)
        r2_cv = float(np.clip(1 - sse_cv / sst_cv, -1.0, 1.0))

    # --- 5. Metrics ---
    metrics = {
        "Dist Code": g["Dist Code"].iat[0],
        "state": g["state"].iat[0],
        "district": g["district"].iat[0],
        "n_years": int(n),

        "R2_1_Trend": r2_1,
        "R2_2_Climate": r2_2,
        "R2_3_Irrig": r2_3,
        "R2_4_Full": r2_4,

        "Mq_Climate": r2_2 - r2_1,
        "Mq_Irrig": r2_3 - r2_2,
        "Mq_SIF": r2_4 - r2_3,

        "R2_cv": r2_cv
    }

    return metrics, None